# Homework

- Use this assignment to practice the chapter's concepts and coding skills in runnable C# code.
- Sign in from the account menu before you start working.
- Write and run your code directly in the coding cells.
- Multiple submissions are allowed. The highest score is kept.


## True or False

```{index} True or False
```


**Question 1.** A case-insensitive customer index treats two spellings as equal keys, so its hash policy must assign them equal hash codes.

<label><input type="radio" name="cscs-hash_collections-homework-tf-1" value="true"> True</label>
<label><input type="radio" name="cscs-hash_collections-homework-tf-1" value="false"> False</label>


```{dropdown} Answer
True. Equal keys require equal hash codes under the same equality policy. Unequal keys may still collide.
```


**Question 2.** Two distinct product keys can occupy the same hash bucket without implying that they name the same product.

<label><input type="radio" name="cscs-hash_collections-homework-tf-2" value="true"> True</label>
<label><input type="radio" name="cscs-hash_collections-homework-tf-2" value="false"> False</label>


```{dropdown} Answer
True. Bucket mapping has fewer positions than possible keys. Chaining or probing resolves the collision using equality.
```


**Question 3.** An export team can safely store GetHashCode output as the permanent business ID used across application restarts.

<label><input type="radio" name="cscs-hash_collections-homework-tf-3" value="true"> True</label>
<label><input type="radio" name="cscs-hash_collections-homework-tf-3" value="false"> False</label>


```{dropdown} Answer
False. Hash codes support in-memory bucket selection and are not stable business identifiers. Store an explicit durable ID.
```


**Question 4.** If branch A and branch B need their missing-product lists, A minus B can differ from B minus A.

<label><input type="radio" name="cscs-hash_collections-homework-tf-4" value="true"> True</label>
<label><input type="radio" name="cscs-hash_collections-homework-tf-4" value="false"> False</label>


```{dropdown} Answer
True. Directional difference returns values only in the first operand. Reverse the operands for the other branch, preserving originals if both results are needed.
```


**Question 5.** An importer that cannot find a product key may automatically record stock zero without distinguishing a missing record.

<label><input type="radio" name="cscs-hash_collections-homework-tf-5" value="true"> True</label>
<label><input type="radio" name="cscs-hash_collections-homework-tf-5" value="false"> False</label>


```{dropdown} Answer
False. Absence and a recorded zero are different states. TryGetValue supplies a presence signal; the business policy decides what to do with missing data.
```


## Coding Practice

```{index} Coding Practice
```


**Question 6: A Missing Key Is Not Zero**

Distinguish absent records from recorded zero values.


In [ ]:
using System;
using System.Collections.Generic;

Dictionary<string, int> stock = new Dictionary<string, int>(); stock["P100"] = 0;
Console.WriteLine(// Your code starts here.
stock.TryGetValue("P100", out int quantity) /* TODO: inspect P200 */
// Your code ends here.
 ? quantity.ToString() : "unrecorded");


In [1]:
using System;
using System.Collections.Generic;

Dictionary<string, int> stock = new Dictionary<string, int>(); stock["P100"] = 0;
Console.WriteLine(stock.TryGetValue("P200", out int quantity) ? quantity.ToString() : "unrecorded");


unrecorded


**Question 7: Combine Branch Membership**

Create the union of two branch subscription sets.


In [ ]:
using System;
using System.Collections.Generic;

HashSet<string> all = new HashSet<string> {"A", "B"};
// Your code starts here.
all.IntersectWith /* TODO: choose union */
// Your code ends here.
(new[] {"B", "C"});
Console.WriteLine(all.Count);


In [1]:
using System;
using System.Collections.Generic;

HashSet<string> all = new HashSet<string> {"A", "B"};
all.UnionWith(new[] {"B", "C"});
Console.WriteLine(all.Count);


3


**Question 8: Find Shared Membership**

Count customers present in both branch sets.


In [ ]:
using System;
using System.Collections.Generic;

HashSet<string> shared = new HashSet<string> {"A", "B"};
// Your code starts here.
shared.UnionWith /* TODO: choose intersection */
// Your code ends here.
(new[] {"B", "C"});
Console.WriteLine(shared.Count);


In [1]:
using System;
using System.Collections.Generic;

HashSet<string> shared = new HashSet<string> {"A", "B"};
shared.IntersectWith(new[] {"B", "C"});
Console.WriteLine(shared.Count);


1


**Question 9: Use Identifier Equality**

Ignore letter case consistently when storing and looking up department IDs.


In [ ]:
using System;
using System.Collections.Generic;

Dictionary<string, int> counts = new Dictionary<string, int>(// Your code starts here.
StringComparer.Ordinal /* TODO: choose identity policy */
// Your code ends here.
);
counts["IT"] = 4;
Console.WriteLine(counts.TryGetValue("it", out int count) ? count : -1);


In [1]:
using System;
using System.Collections.Generic;

Dictionary<string, int> counts = new Dictionary<string, int>(StringComparer.OrdinalIgnoreCase);
counts["IT"] = 4;
Console.WriteLine(counts.TryGetValue("it", out int count) ? count : -1);


4


**Question 10: Preserve Map Count on Update**

Updating an existing ticket changes its value without adding a new key.


In [ ]:
using System;
using System.Collections.Generic;

TicketMap map = new TicketMap();
map.Put(3, "Open"); // Your code starts here.
/* TODO: replace the value for the existing key. */
// Your code ends here.

Console.WriteLine(map.Count);
Console.WriteLine(map.TryGetValue(3, out string status) ? status : "missing");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
        {
            var ids = new List<int>();
            foreach (Entry entry in buckets[i]) ids.Add(entry.Id);
            lines.Add($"{i}: {string.Join(", ", ids)}");
        }
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}


In [1]:
using System;
using System.Collections.Generic;

TicketMap map = new TicketMap();
map.Put(3, "Open"); map.Put(3, "Closed");
Console.WriteLine(map.Count);
Console.WriteLine(map.TryGetValue(3, out string status) ? status : "missing");

public class TicketMap
{
    private List<Entry>[] buckets = CreateBuckets(4);
    public int Count { get; private set; }
    public int BucketCount => buckets.Length;
    public double LoadFactor => (double)Count / buckets.Length;

    public bool Put(int id, string owner)
    {
        int index = Bucket(id, buckets.Length);
        for (int i = 0; i < buckets[index].Count; i++)
        {
            if (buckets[index][i].Id == id)
            {
                buckets[index][i] = new Entry(id, owner);
                return false;
            }
        }
        if ((Count + 1.0) / buckets.Length > 0.75)
        {
            Grow();
            index = Bucket(id, buckets.Length);
        }
        buckets[index].Add(new Entry(id, owner));
        Count++;
        return true;
    }

    public bool TryGetValue(int id, out string owner)
    {
        foreach (Entry entry in buckets[Bucket(id, buckets.Length)])
        {
            if (entry.Id == id) { owner = entry.Owner; return true; }
        }
        owner = default!;
        return false;
    }

    public bool Remove(int id)
    {
        List<Entry> chain = buckets[Bucket(id, buckets.Length)];
        for (int i = 0; i < chain.Count; i++)
        {
            if (chain[i].Id != id) continue;
            chain.RemoveAt(i);
            Count--;
            return true;
        }
        return false;
    }

    private void Grow()
    {
        List<Entry>[] replacement = CreateBuckets(buckets.Length * 2);
        foreach (List<Entry> chain in buckets)
            foreach (Entry entry in chain)
                replacement[Bucket(entry.Id, replacement.Length)].Add(entry);
        buckets = replacement;
    }

    private static int Bucket(int id, int length) => (int)(unchecked((uint)id) % (uint)length);

    private static List<Entry>[] CreateBuckets(int length)
    {
        var result = new List<Entry>[length];
        for (int i = 0; i < length; i++) result[i] = new List<Entry>();
        return result;
    }

    public string DescribeBuckets()
    {
        var lines = new List<string>();
        for (int i = 0; i < buckets.Length; i++)
        {
            var ids = new List<int>();
            foreach (Entry entry in buckets[i]) ids.Add(entry.Id);
            lines.Add($"{i}: {string.Join(", ", ids)}");
        }
        return string.Join(" | ", lines);
    }

    private class Entry
    {
        public int Id { get; }
        public string Owner { get; }
        public Entry(int id, string owner) { Id = id; Owner = owner; }
    }
}


1
Closed


## Trace Collisions and Maintain Coordinated Indexes



**Independent assessment.** Use separate chaining with bucket index `key % capacity`, initial capacity four, and a growth rule that doubles capacity before insertion would make load exceed 0.75. Trace insertions 1, 5, 9, and 13, replacement of key 5, removal of key 9, and successful/failed lookup. Show every bucket before and after rehashing; count comparisons. Replacement must not increase Count. For a separate open-addressed table of capacity seven using linear probing, insert 1, 8, and 15, then remove 8. Show why lookup of 15 must continue past a tombstone and where an absent-key search stops. Do not treat these two representations as one deletion policy.

Next implement a cancellation-aware service queue using a FIFO queue of unique ticket IDs plus a dictionary of active IDs. Cancellation removes the dictionary entry; service skips stale queue entries and removes the first active ticket from both logical views. Reject duplicate active IDs. For this task IDs are never reused, even after cancellation, so an old queue entry cannot accidentally refer to a newly issued ticket. Add an issued-ID set to enforce that contract. Test cancel-first, cancel-all, duplicate issue, reissue after cancellation, and empty service. Explain which invariant coordinates the indexes and why updating only one is incorrect. Justify this representation for frequent lookup/cancellation and FIFO service; compare a workload requiring sorted range queries.


Finally, change the service policy to smallest urgency first, with arrival order breaking equal urgencies. Coordinate a `PriorityQueue` with an active-ticket dictionary and a monotonically increasing version for each changed ticket. Reprioritizing a ticket adds a new queue entry; an older version must never serve the updated ticket. Cancellation removes the active record; dequeue skips entries whose ID is absent or version is stale. Retain the issued-ID policy. Test duplicate issue, two successive priority changes, equal-urgency arrival order, cancellation of an urgent ticket, cancel-all, and empty service. Verify that each active ticket is served at most once with its current urgency/title, and explain why dictionary presence alone cannot detect a superseded entry. State the expected dictionary lookup cost and the heap operation cost separately.


In [ ]:
// Exercise: Trace Collisions and Maintain Coordinated Indexes
// Write your implementation and your own checks below.
// TODO: implement the contract in the question.


In [1]:
// One solution. Key-comparison counters count equality checks in occupied slots only.
using System;
using System.Collections.Generic;
using System.Linq;

List<KeyValuePair<int, string>>[] MakeBuckets(int capacity)
{
    var result = new List<KeyValuePair<int, string>>[capacity];
    for (int i = 0; i < capacity; i++) result[i] = new();
    return result;
}
var buckets = MakeBuckets(4);
int mapCount = 0, keyComparisons = 0;
void ShowBuckets(string label)
{
    Console.WriteLine(label + ": " + string.Join(" | ", buckets.Select((b, i) =>
        i + "=[" + string.Join(",", b.Select(p => $"{p.Key}:{p.Value}")) + "]")));
}
int Position(int key)
{
    var bucket = buckets[key % buckets.Length];
    for (int i = 0; i < bucket.Count; i++)
    { keyComparisons++; if (bucket[i].Key == key) return i; }
    return -1;
}
void Put(int key, string value)
{
    int before = keyComparisons, position = Position(key);
    if (position >= 0) buckets[key % buckets.Length][position] = new(key, value);
    else
    {
        if ((mapCount + 1) * 4 > buckets.Length * 3)
        {
            ShowBuckets("before rehash");
            var old = buckets; buckets = MakeBuckets(old.Length * 2);
            foreach (var bucket in old)
                foreach (var pair in bucket) buckets[pair.Key % buckets.Length].Add(pair);
            ShowBuckets("after rehash");
        }
        buckets[key % buckets.Length].Add(new(key, value)); mapCount++;
    }
    ShowBuckets($"put {key}, Count={mapCount}, comparisons={keyComparisons - before}");
}
bool RemoveKey(int key)
{
    int before = keyComparisons, position = Position(key);
    if (position < 0) return false;
    buckets[key % buckets.Length].RemoveAt(position); mapCount--;
    ShowBuckets($"remove {key}, Count={mapCount}, comparisons={keyComparisons - before}");
    return true;
}
bool Lookup(int key)
{
    int before = keyComparisons; bool found = Position(key) >= 0;
    Console.WriteLine($"lookup {key}: found={found}, comparisons={keyComparisons - before}");
    return found;
}
foreach (int key in new[] { 1, 5, 9, 13 }) Put(key, $"V{key}");
Put(5, "updated");
if (mapCount != 4 || !RemoveKey(9) || !Lookup(5) || Lookup(9))
    throw new Exception("Chaining contract failed.");
Console.WriteLine($"chaining total key comparisons={keyComparisons}");

// Open addressing has a separate deletion policy: 0 empty, 1 live, 2 tombstone.
int[] keys = new int[7], state = new int[7];
void ProbeInsert(int key)
{
    for (int step = 0; step < keys.Length; step++)
    {
        int slot = (key % keys.Length + step) % keys.Length;
        if (state[slot] == 1) continue;
        keys[slot] = key; state[slot] = 1; return;
    }
    throw new InvalidOperationException("Probe table full.");
}
bool ProbeFind(int key)
{
    List<string> trace = new(); int comparisons = 0;
    for (int step = 0; step < keys.Length; step++)
    {
        int slot = (key % keys.Length + step) % keys.Length;
        trace.Add($"{slot}:{(state[slot] == 0 ? "empty" : state[slot] == 2 ? "deleted" : keys[slot].ToString())}");
        if (state[slot] == 0)
        { Console.WriteLine($"probe {key}: {string.Join(" -> ", trace)}, found=False, comparisons={comparisons}"); return false; }
        if (state[slot] == 2) continue;
        comparisons++;
        if (keys[slot] == key)
        { Console.WriteLine($"probe {key}: {string.Join(" -> ", trace)}, found=True, comparisons={comparisons}"); return true; }
    }
    return false;
}
foreach (int key in new[] { 1, 8, 15 }) ProbeInsert(key);
Console.WriteLine("probe slots before delete: " + string.Join(",", state.Select((s, i) => s == 1 ? $"{i}:{keys[i]}" : $"{i}:empty")));
state[2] = 2; // key 8 occupied slot 2.
Console.WriteLine("probe slots after delete: " + string.Join(",", state.Select((s, i) => s == 1 ? $"{i}:{keys[i]}" : s == 2 ? $"{i}:deleted" : $"{i}:empty")));
if (!ProbeFind(15) || ProbeFind(22)) throw new Exception("Tombstone contract failed.");

// FIFO variant: never reusing IDs makes cancellation-by-presence sufficient.
Queue<int> waiting = new();
Dictionary<int, string> active = new();
HashSet<int> issued = new();
bool Issue(int id, string title)
{
    if (!issued.Add(id)) return false;
    active.Add(id, title); waiting.Enqueue(id); return true;
}
bool Serve(out string title)
{
    title = "";
    while (waiting.Count > 0)
    {
        int id = waiting.Dequeue();
        if (active.Remove(id, out string? found)) { title = found; return true; }
    }
    return false;
}
Issue(1, "North"); Issue(2, "South"); active.Remove(1);
if (Issue(1, "Reused") || Issue(2, "Duplicate") || !Serve(out string title) || title != "South" || Serve(out title))
    throw new Exception("FIFO contract failed.");
Issue(3, "East"); Issue(4, "West"); active.Clear();
if (Serve(out title) || active.Count != 0) throw new Exception("FIFO cancel-all failed.");
Console.WriteLine("FIFO: duplicate/reissue rejected; cancelled IDs skipped; cancel-all and empty pass");

// Priority variant: a presence check alone cannot distinguish superseded entries.
var priority = new PriorityQueue<(int Id, long Version), (int Urgency, long Arrival)>();
var current = new Dictionary<int, (string Title, int Urgency, long Version, long Arrival)>();
var priorityIssued = new HashSet<int>();
long nextArrival = 0, nextVersion = 0;
bool OpenTicket(int id, string title, int urgency)
{
    if (!priorityIssued.Add(id)) return false;
    long arrival = nextArrival++, version = ++nextVersion;
    current.Add(id, (title, urgency, version, arrival));
    priority.Enqueue((id, version), (urgency, arrival)); return true;
}
bool Reprioritize(int id, string title, int urgency)
{
    if (!current.TryGetValue(id, out var ticket)) return false;
    long version = ++nextVersion;
    current[id] = (title, urgency, version, ticket.Arrival);
    priority.Enqueue((id, version), (urgency, ticket.Arrival)); return true;
}
bool Dispatch(out int id, out string title, out int urgency)
{
    id = -1; title = ""; urgency = 0;
    while (priority.TryDequeue(out var entry, out var queuedPriority))
    {
        if (!current.TryGetValue(entry.Id, out var ticket))
        { Console.WriteLine($"skip cancelled/served {entry.Id}"); continue; }
        if (entry.Version != ticket.Version)
        { Console.WriteLine($"skip superseded {entry.Id}: queued={entry.Version}, current={ticket.Version}"); continue; }
        current.Remove(entry.Id); id = entry.Id; title = ticket.Title; urgency = ticket.Urgency;
        Console.WriteLine($"serve {id}:{title}, urgency={urgency}, arrival={ticket.Arrival}"); return true;
    }
    return false;
}
OpenTicket(10, "Old", 0); OpenTicket(20, "Second", 1); OpenTicket(30, "Cancelled", -1);
if (OpenTicket(10, "Duplicate", -2)) throw new Exception("Duplicate active ID accepted.");
Reprioritize(10, "Middle", 1); Reprioritize(10, "Current", 2); current.Remove(30);
if (!Dispatch(out int id, out title, out int urgency) || id != 20 || urgency != 1)
    throw new Exception("Superseded priority served before a more urgent current ticket.");
if (!Dispatch(out id, out title, out urgency) || id != 10 || title != "Current" || urgency != 2 || Dispatch(out id, out title, out urgency))
    throw new Exception("Current-version or at-most-once dispatch failed.");
OpenTicket(60, "First equal", 3); OpenTicket(70, "Second equal", 3);
if (!Dispatch(out id, out title, out urgency) || id != 60 || !Dispatch(out id, out title, out urgency) || id != 70)
    throw new Exception("Equal-priority arrival order failed.");
OpenTicket(40, "Cancel all A", 0); OpenTicket(50, "Cancel all B", 0); current.Clear();
if (Dispatch(out id, out title, out urgency) || OpenTicket(30, "Reused", 0) || Reprioritize(99, "Absent", 0))
    throw new Exception("Cancel-all, issued-ID, or absent-update contract failed.");
Console.WriteLine("priority: superseded/cancelled skipped; arrival ties, at-most-once, duplicate/reissue, cancel-all, and empty pass");
Console.WriteLine("Expected dictionary lookup O(1); heap enqueue/dequeue O(log q); stale entries add removals, so bound complete workload by all queued versions.");


put 1, Count=1, comparisons=0: 0=[] | 1=[1:V1] | 2=[] | 3=[]
put 5, Count=2, comparisons=1: 0=[] | 1=[1:V1,5:V5] | 2=[] | 3=[]
put 9, Count=3, comparisons=2: 0=[] | 1=[1:V1,5:V5,9:V9] | 2=[] | 3=[]
before rehash: 0=[] | 1=[1:V1,5:V5,9:V9] | 2=[] | 3=[]
after rehash: 0=[] | 1=[1:V1,9:V9] | 2=[] | 3=[] | 4=[] | 5=[5:V5] | 6=[] | 7=[]
put 13, Count=4, comparisons=3: 0=[] | 1=[1:V1,9:V9] | 2=[] | 3=[] | 4=[] | 5=[5:V5,13:V13] | 6=[] | 7=[]
put 5, Count=4, comparisons=1: 0=[] | 1=[1:V1,9:V9] | 2=[] | 3=[] | 4=[] | 5=[5:updated,13:V13] | 6=[] | 7=[]
remove 9, Count=3, comparisons=2: 0=[] | 1=[1:V1] | 2=[] | 3=[] | 4=[] | 5=[5:updated,13:V13] | 6=[] | 7=[]
lookup 5: found=True, comparisons=1
lookup 9: found=False, comparisons=1
chaining total key comparisons=11
probe slots before delete: 0:empty,1:1,2:8,3:15,4:empty,5:empty,6:empty
probe slots after delete: 0:empty,1:1,2:deleted,3:15,4:empty,5:empty,6:empty
probe 15: 1:1 -> 2:deleted -> 3:15, found=True, comparisons=2
probe 22: 1:1 -> 2:delet

```{dropdown} Explain the trace and the two coordination contracts
At capacity four, keys 1,5,9 share bucket one. Inserting 13 would exceed load 0.75, so rehash first: at capacity eight, bucket one holds 1,9 and bucket five holds 5, then insertion adds 13 to bucket five. Replacing key 5 keeps Count four; removing 9 reduces it to three. The reference code prints every bucket and counts occupied-slot key equality checks separately from copying during rehash.

Linear probing places 1,8,15 in slots 1,2,3 of capacity seven. Deleting 8 leaves a tombstone at two: lookup 15 continues to three, and absent 22 continues to the empty slot four. Marking slot two empty would incorrectly end lookup before 15.

The FIFO variant forbids ID reuse and therefore uses active membership to skip cancellations. A priority update creates a different problem: an ID can remain active while an old queue entry is obsolete. The priority variant stores a version with each entry and compares it with the active dictionary record. Its test reprioritizes ticket 10 twice and leaves ticket 20 more urgent than 10's current version; serving an obsolete priority for 10 would violate the asserted order. Separate equal-urgency tests check arrival order, and cancellation/empty/duplicate checks exercise the remaining boundaries.

The logical invariant is that the dictionary contains the current active records; the queue may additionally contain obsolete physical entries. Dequeue must skip those entries, then remove exactly the served active record. Dictionary access is expected constant work under suitable hashing; heap operations depend on queue size, with occasional storage growth and all stale-version removals included in a complete workload. A FIFO queue suits arrival-order service; an ordered tree/sorted representation may suit range queries. A hash map alone does not provide ordered ranges.
```


## Submit

```{index} Submit
```


Use the course assignment panel to submit your answers. Sign in before submitting; multiple attempts are allowed and the highest score is kept. Keep your completed code and explain any failed check before resubmitting. In a local static preview, use the notebook or your saved console project to retain your work; this page alone does not submit to the course service.
